# 1) Inference

In [21]:
# test code 
import pandas as pd
from openai import OpenAI
import os
import time

# set API key
os.environ["OPENAI_API_KEY"] = "API_KEY"
client = OpenAI()

# load dataset
df = pd.read_csv("dataset_clean.csv")

results = []

# test with 3 rows
for index, row in df.head(3).iterrows():
    question_id = row["id"]
    question = row["prompt"]

    print(f"Processing {question_id}...")

    # send a request to the OpenAI Chat Completion API
    response = client.chat.completions.create(
        model="gpt-4o-mini",

        # provide conversation context as a list of messages
        messages=[
            {
                # system role defines how the assistant should behave
                "role": "system",
                 # instruction to the model
                "content": "You are an expert in Austrian corporate tax law. Answer precisely, completely, and include legal references (e.g. § ... KStG) if possible."
            },
            {
                # user role contains the actual question that should be answered
                "role": "user",
                # variable 'question' stores the current question being processed
                "content": question
            }
        ],
        # temperature=0 makes output deterministic
        temperature=0,
        # limits maximum length of the generated answer
        max_tokens=400   # prevents cutoff
    )

    # extract generated answer text from the API response
    answer = response.choices[0].message.content

    print(answer)
    # print separator line for multiple outputs
    print("-----")

    results.append({
        "id": question_id,
        "answer": answer
    })

# save test output
pd.DataFrame(results).to_csv("test_output.csv", index=False)

print("DONE")

Processing CORP-TAX-001...
Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer in Österreich wird durch den Gewinn des Unternehmens bestimmt. Gemäß § 7 des Körperschaftsteuergesetzes (KStG) ist der Gewinn der Betrag, der sich aus der Gewinnermittlung nach den Vorschriften des Unternehmensgesetzbuches (UGB) ergibt, wobei steuerliche Anpassungen vorzunehmen sind.

Die wichtigsten Punkte zur Ermittlung der Bemessungsgrundlage sind:

1. **Gewinnermittlung**: Der Gewinn wird in der Regel durch die Bilanzierung ermittelt, die nach den Vorschriften des UGB erfolgt. Hierbei sind sowohl die Einnahmen als auch die Ausgaben zu berücksichtigen.

2. **Steuerliche Anpassungen**: Es sind verschiedene steuerliche Anpassungen vorzunehmen, um den steuerlichen Gewinn zu ermitteln. Dazu gehören beispielsweise:
   - Hinzurechnungen (§ 8 KStG): Bestimmte Aufwendungen, die steuerlich nicht abzugsfähig sind, müssen dem Gewinn wieder hinzugerechnet werden.
   - Kürzungen (§ 9 KStG): Bestimmte Einnah

In [20]:
import pandas as pd

df = pd.read_csv("dataset_clean.csv")

In [23]:
# just 
df = pd.read_csv("dataset_clean.csv")

results = []

for index, row in df.iterrows():
    question_id = row["id"]
    question = row["prompt"]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert in Austrian tax law."},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=300
    )

    results.append({
        "id": question_id,
        "answer": response.choices[0].message.content
    })

    # small delay to avoid rate limits
    time.sleep(0.3)

In [24]:
import pandas as pd

df_results = pd.DataFrame(results)

df_results.to_csv("model1_inference.csv", index=False)

In [25]:
# check answers and df
print(df_results.head())
print(len(df_results))   # MUST be 644

             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Die Vergabe eines zinslosen Darlehens von eine...
2  CORP-TAX-003  In Österreich sind bestimmte Körperschaften ve...
3  CORP-TAX-004  In Österreich wird die steuerliche Behandlung ...
4  CORP-TAX-005  In der österreichischen Steuerrechtsprechung g...
643


In [27]:
# read and check outputs

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("model1_inference.csv")
df.head(2)

,id,answer
0,CORP-TAX-001,"Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer in Österreich ist der Gewinn eines Unternehmens. Dieser Gewinn wird ermittelt, indem die Betriebseinnahmen um die Betriebsausgaben reduziert werden. \n\nDie wichtigsten Schritte zur Ermittlung der Körperschaftsteuer-Bemessungsgrundlage sind:\n\n1. **Ermittlung des Jahresgewinns**: Der Jahresgewinn wird aus der Gewinn- und Verlustrechnung (GuV) abgeleitet. Hierbei werden alle Einnahmen und Ausgaben des Unternehmens berücksichtigt.\n\n2. **Berücksichtigung von steuerlichen Anpassungen**: Es gibt bestimmte steuerliche Vorschriften, die Anpassungen am Gewinn erfordern können. Dazu gehören beispielsweise:\n - Nicht abzugsfähige Betriebsausgaben (z.B. bestimmte Repräsentationskosten)\n - Steuerfreie Einnahmen\n - Abweichungen zwischen handelsrechtlicher und steuerlicher Behandlung von Rückstellungen oder Abschreibungen\n\n3. **Verlustvortrag und Verlustabzug**: Verluste aus Vorjahren können unter bestimmten Voraussetzungen mit Gewinnen aus dem laufenden Jahr verrechnet werden, was die Bemessungsgrundlage für die Körperschaftsteuer verringern kann.\n\n4. **Steuersatz**: Der Gewinn, der nach diesen Anpassungen ermittelt wurde, unterliegt dem Körperschaftsteuersatz, der in Österreich derzeit 25 % beträgt.\n\nEs ist wichtig, die spezifischen Vorschriften und Regelungen im österreichischen Körperschaftsteuergesetz ("
1,CORP-TAX-002,"Die Vergabe eines zinslosen Darlehens von einer Körperschaft an ihren Gesellschafter hat in Österreich mehrere steuerliche Konsequenzen, die sowohl die Körperschaft als auch den Gesellschafter betreffen können. Hier sind die wesentlichen Punkte:\n\n1. **Verdeckte Gewinnausschüttung**: Wenn eine Körperschaft ein zinsloses Darlehen an einen Gesellschafter vergibt, kann dies als verdeckte Gewinnausschüttung (vGA) qualifiziert werden. Eine vGA liegt vor, wenn der Gesellschafter einen Vorteil erhält, der nicht im Rahmen eines ordnungsgemäßen Geschäftsverkehrs erfolgt. In diesem Fall könnte die Körperschaft verpflichtet sein, die vGA als steuerpflichtigen Gewinn zu erfassen.\n\n2. **Zinsaufwand**: Da das Darlehen zinslos ist, könnte die Körperschaft auch einen fiktiven Zinsaufwand ansetzen müssen. Dieser fiktive Zins wird in der Regel auf Basis eines marktüblichen Zinssatzes berechnet. Der Zinsaufwand würde die steuerliche Bemessungsgrundlage der Körperschaft mindern.\n\n3. **Einkommensteuer des Gesellschafters**: Der Gesellschafter könnte ebenfalls steuerliche Konsequenzen haben. Wenn das zinslose Darlehen als vGA betrachtet wird, könnte dies zu einer Erhöhung des steuerpflichtigen Einkommens des Gesellschafters führen, was"
